# 1. CARGA Y VISION GENERAL

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', None)

# Cargar el dataset
df = pd.read_csv('data/train.csv')
print(f"Shape: {df.shape}")
display(df.head())

# Eliminar columnas de ID si existen
df.drop(['Id'], axis=1, inplace=True, errors='ignore')

# 2. ANALISIS DEL TARGET (SalePrice)

In [ ]:
# Histograma de la distribución original del target
target = 'SalePrice'
plt.figure(figsize=(8,4))
sns.histplot(df[target], kde=True)
plt.title('Distribución original de SalePrice')
plt.show()

# Comprobar si es continuo o categórico
target_unique = df[target].nunique()
if target_unique > 20:
    # Variable continua
    print(f"Sesgo (skew): {df[target].skew():.3f}")
    # Si está sesgado, probar transformación logarítmica
df['log_SalePrice'] = np.log(df[target])
    
    plt.figure(figsize=(8,4))
    sns.histplot(df['log_SalePrice'], kde=True)
    plt.title('SalePrice tras transformación logarítmica')
    plt.show()
else:
    # Variable categórica
    print(df[target].value_counts())
    plt.figure(figsize=(8,4))
    sns.countplot(y=df[target])
    plt.title('Balance de clases SalePrice')
    plt.show()

# 3. TIPOS DE VARIABLES

In [ ]:
# Variables categóricas (objeto)
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

# Numéricas disfrazadas de categóricas
disguised_cat = ['MSSubClass'] if 'MSSubClass' in df.columns else []
for col in disguised_cat:
    df[col] = df[col].astype('O')
cat_cols += disguised_cat
cat_cols = list(sorted(set(cat_cols)))

# Variables numéricas (excluye target y categóricas)
num_cols = [col for col in df.columns if df[col].dtype in [np.int64, np.float64] 
            and col not in cat_cols + [target]]

print(f'Numéricas: {len(num_cols)}; Categóricas: {len(cat_cols)}; Target: {target}')
print('Numéricas:', num_cols)
print('Categóricas:', cat_cols)

# 4. MISSING VALUES

In [ ]:
missing_perc = df.isnull().mean().sort_values(ascending=False) * 100
vars_with_missing = missing_perc[missing_perc > 0].index.tolist()

print('Variables con missing values:', missing_perc[missing_perc > 0])
plt.figure(figsize=(10,5))
sns.barplot(y=missing_perc[missing_perc>0].index, x=missing_perc[missing_perc>0].values)
plt.title('Porcentaje de missing values por variable')
plt.xlabel('% missing')
plt.show()

cat_na = [col for col in vars_with_missing if col in cat_cols]
num_na = [col for col in vars_with_missing if col in num_cols]
print('Categóricas con NA:', cat_na)
print('Numéricas con NA:', num_na)

# Relación missing vs. SalePrice
for col in vars_with_missing:
    miss_mask = df[col].isnull().astype(int)
    means = df.groupby(miss_mask)[target].mean()
    plt.figure(figsize=(4,3))
    sns.barplot(x=means.index, y=means.values)
    plt.title(f'{col}: Mean SalePrice by Missingness (1=missing)')
    plt.xlabel('Missing')
    plt.ylabel('Mean SalePrice')
    plt.show()

# 5. VARIABLES NUMERICAS

## 5a. VARIABLES TEMPORALES

In [ ]:
temp_cols = [col for col in num_cols if any(s in col for s in ['Yr', 'Year'])]
print('Variables temporales detectadas:', temp_cols)

for col in temp_cols:
    plt.figure(figsize=(8,3))
    sns.lineplot(x=col, y=target, data=df.groupby(col)[target].median().reset_index())
    plt.title(f'Mediana de SalePrice vs {col}')
    plt.show()

ref_year_col = 'YrSold' if 'YrSold' in df.columns else None
years_elapsed = {}
if ref_year_col:
    for col in temp_cols:
        if col != ref_year_col:
            new_col = f'{col}_elapsed'
            df[new_col] = df[ref_year_col] - df[col]
            years_elapsed[new_col] = col
            plt.figure(figsize=(8,3))
            sns.scatterplot(x=df[new_col], y=np.log(df[target]))
            plt.title(f'{new_col} vs log(SalePrice)')
            plt.show()

## 5b. VARIABLES DISCRETAS

In [ ]:
discrete_cols = [col for col in num_cols if df[col].nunique()<20 and col not in temp_cols]
print('Variables discretas:', discrete_cols)

for col in discrete_cols:
    plt.figure(figsize=(8,3))
    sns.boxplot(x=col, y=target, data=df)
    sns.stripplot(x=col, y=target, data=df, color='orange', alpha=0.2)
    plt.title(f'{col} vs SalePrice')
    plt.show()
    # Podría agregarse análisis de relación creciente

## 5c. VARIABLES CONTINUAS

In [ ]:
continuous_cols = [col for col in num_cols if col not in discrete_cols + temp_cols]
print('Variables continuas:', continuous_cols)

from sklearn.preprocessing import PowerTransformer
for col in continuous_cols:
    fig, axes = plt.subplots(1,2, figsize=(12,4))
    sns.histplot(df[col].dropna(), kde=True, ax=axes[0])
    axes[0].set_title(f'{col}: Original')
    # Transf Yeo-Johnson
    pt = PowerTransformer(method='yeo-johnson')
    arr = df[[col]].dropna()
    arr_t = pt.fit_transform(arr)
    sns.histplot(arr_t.ravel(), kde=True, ax=axes[1])
    axes[1].set_title(f'{col}: Yeo-Johnson')
    plt.tight_layout()
    plt.show()
    # Log (si no tiene ceros)
    if (df[col]>0).all():
        plt.figure(figsize=(6,2))
        sns.histplot(np.log(df[col]), kde=True)
        plt.title(f'{col}: log-transform')
        plt.show()
    plt.figure(figsize=(8,3))
    sns.scatterplot(x=df[col], y=df['log_SalePrice'])
    plt.title(f'{col} vs log(SalePrice)')
    plt.show()
    if (df[col]>0).all():
        plt.figure(figsize=(8,3))
        sns.scatterplot(x=np.log(df[col]), y=df['log_SalePrice'])
        plt.title(f'log({col}) vs log(SalePrice)')
        plt.show()

## 5d. SESGADAS / DOMINADAS POR CEROS

In [ ]:
skewed_sparse = ['MasVnrArea','BsmtFinSF2','2ndFlrSF','LowQualFinSF','BsmtFullBath','BsmtHalfBath','HalfBath','WoodDeckSF','EnclosedPorch','3SsnPorch','ScreenPorch','PoolArea','MiscVal']
for col in skewed_sparse:
    df[f'{col}_iszero'] = (df[col]==0).astype(int)
    means = df.groupby(f'{col}_iszero')[target].mean()
    plt.figure(figsize=(4,3))
    sns.barplot(x=means.index, y=means.values)
    plt.title(f'Mean SalePrice by is_zero: {col}')
    plt.xlabel('IsZero')
    plt.ylabel('Mean SalePrice')
    plt.show()

# 6. VARIABLES CATEGORICAS

## 6a. CARDINALIDAD

In [ ]:
cat_card = {col: df[col].nunique() for col in cat_cols}
plt.figure(figsize=(10,4))
sns.barplot(x=list(cat_card.keys()), y=list(cat_card.values()))
plt.xticks(rotation=60, ha='right')
plt.title('Cardinalidad de variables categóricas')
plt.ylabel('Nº categorías únicas')
plt.show()

high_card = [col for col, n in cat_card.items() if n > 10]
print('Alta cardinalidad:', high_card)

## 6b. VARIABLES ORDINALES DE CALIDAD

In [ ]:
ordinal_quality_map = {
    'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1
}
ordinal_vars = [col for col in cat_cols if set(df[col].dropna().unique()).issubset(ordinal_quality_map.keys())]
print('Variables ordinales de calidad:', ordinal_vars)
for col in ordinal_vars:
    mapped = df[col].map(ordinal_quality_map)
    df[col+'_num'] = mapped
    plt.figure(figsize=(6,2))
    sns.boxplot(x=df[col+'_num'], y=df[target])
    plt.title(f'{col} (mapeado) vs SalePrice')
    plt.show()

## 6c. RARE LABELS

In [ ]:
for col in cat_cols:
    freqs = df[col].value_counts(normalize=True)
    rare = freqs[freqs < 0.01]
    if not rare.empty:
        print(f'Variable {col}: rare labels ->', rare.index.tolist())

## 6d. CATEGORICAS NOMINALES vs TARGET

In [ ]:
nominals = [col for col in cat_cols if col not in ordinal_vars]
for col in nominals:
    if df[col].nunique() < 15:
        plt.figure(figsize=(max(5,df[col].nunique()),3))
        sns.boxplot(x=df[col], y=df[target])
        sns.stripplot(x=df[col], y=df[target], color='orange', alpha=0.2)
        plt.title(f'{col} vs SalePrice')
        plt.xticks(rotation=45)
        plt.show()

# 7. CONCLUSIONES

Variables que necesitan imputación:
- Numéricas: LotFrontage, GarageYrBlt, MasVnrArea
- Categóricas: PoolQC, MiscFeature, Alley, Fence, MasVnrType, FireplaceQu, GarageType, GarageFinish, GarageQual, GarageCond, BsmtFinType2, BsmtExposure, BsmtFinType1, BsmtCond, BsmtQual, Electrical

Variables temporales detectadas y variable de referencia:
- Temporales: YearBuilt, YearRemodAdd, GarageYrBlt
- Referencia: YrSold

Variables continuas candidatas a log/Yeo-Johnson:
- LotArea, 1stFlrSF, GrLivArea, SalePrice, MasVnrArea, BsmtFinSF1, TotalBsmtSF

Variables sesgadas candidatas a binarización:
- MasVnrArea, BsmtFinSF2, 2ndFlrSF, LowQualFinSF, BsmtFullBath, BsmtHalfBath, HalfBath, WoodDeckSF, EnclosedPorch, 3SsnPorch, ScreenPorch, PoolArea, MiscVal

Variables ordinales y mappings sugeridos:
- ExterQual, ExterCond, BsmtQual, BsmtCond, HeatingQC, KitchenQual, FireplaceQu, GarageQual, GarageCond
- Mapeo: {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1}

Variables con rare labels candidatas a RareLabelEncoder:
- Todas con categorías <1% de frecuencia (ver outputs 6c)

Variables a eliminar:
- Id (ya eliminada)